In [1]:
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import cmcrameri.cm as cmc
import pyDEW_functions as pDF
import plotting_functions as plF
import pyDEW
import warnings
from thermoengine import model
from copy import deepcopy

# pyDEW flags up a few things when the system is created - this counteracts it
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.options.mode.chained_assignment = None

# Folder to export figures
if not os.path.exists("./figures"):
    os.makedirs("./figures")

In [2]:
# Adjust here the basis species in the DEW system - deepcopy to avoid mucking up defaults
defaultBasisSpecies = pyDEW.defaultsystem.basis_species_names
basisSpecies = deepcopy(defaultBasisSpecies)

defaultOtherSpecies = pyDEW.defaultsystem.other_species_names
otherSpecies = deepcopy(defaultOtherSpecies)

# Add missing Mg complex to the other species
otherSpecies.append('MG(OH)2(AQ)')

# Switch basis species - replace CO2(AQ) with CO3-2 directly to avoid pyDEW issues
basisSpecies[basisSpecies.index('CO2(AQ)')] = 'CO3-2'

# Create DEW system
dew_system = pyDEW.System(basis_species=basisSpecies, other_species=otherSpecies)

In [3]:

# List of reactants formed from mineral dictionaries
olvDict = {'reactant': 'OLIVINE(SS)',
           'moles': 5.9,
           'composition': {'FORSTERITE': 0.9, 'FAYALITE': 0.1}}

opxDict = {'reactant': 'ORTHOPYROXENE(SS)',
           'moles': 1.9,
           'composition': {'ENSTATITE_OR': 0.9, 'FERROSILITE': 0.1}}

cpxDict = {'reactant': 'CLINOPYROXENE(SS)',
           'moles': 0.2,
           'composition': {'DIOPSIDE': 0.9, 'HEDENBERGITE': 0.1}}

spnDict = {'reactant': 'SPINEL',
           'moles': 0.04}

# mgtDict = {'reactant': 'MAGNETITE',
#            'moles': 0.04}

startingReactant = [olvDict, opxDict, cpxDict, spnDict] #, mgtDict]

In [4]:
inputDF = pd.read_excel('data/04_Cascadia_5_mix_VAR.xlsx', sheet_name='starting_solid')
rhos = []
for i in range(len(inputDF)):
    rhos.append(pDF. findReactantDensity(startingReactant, dew_system, inputDF['Pressure (bar)'].iloc[i],
                                         inputDF['Temperature (K)'].iloc[i]))
testDict = {'P (kbar)': [i/1000 for i in inputDF['Pressure (bar)']],
            'density (g/cm3)': rhos}
testDF = pd.DataFrame(testDict)
testDF.to_csv('5_density.csv')

In [7]:
slices = []
for i in range(len(inputDF)):
    volReactant = pDF.reactantToVolDF(startingReactant, dew_system,
                                      inputDF['Pressure (bar)'].iloc[i],
                                      inputDF['Temperature (K)'].iloc[i])
    slices.append(volReactant)

timeZeroDF = pd.DataFrame(slices)
timeZeroDF.to_csv('timeZero.csv')

ValueError: Must pass 2-d input. shape=(193, 1, 7)